## Evaluation harness

Runs both architectures over the persona set and records the outcome of each
run for RQ2 and RQ3.

For each persona the harness records whether the run completed, whether the
recommendation respected the budget ceiling and the statutory bedroom floor,
the latency and number of model calls, and the recommendation itself for the
matched-pair comparison.

Development runs use a five-persona subset so that iteration takes minutes
rather than hours. The full run covers all forty personas with three
repetitions, as specified in Section 3.4.2.

In [1]:
import sys, json, time
import pandas as pd

In [2]:
sys.path.append('../src/agents')

In [3]:
from orchestrator import recommend
from single_agent_baseline import run_baseline

with open('../data/processed/personas.json') as f:
    personas = json.load(f)

print("Personas loaded:", len(personas))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Personas loaded: 40


In [4]:
# Each persona is evaluated against a candidate property. The asking rent is
# set at the 30th percentile market benchmark for their area and entitlement,
# so the case is neither trivially affordable nor hopeless.
from affordability_agent import (
    brma_for_district, council_tax_for_district, LHA_CATEGORY, MARKET_30TH, weekly_to_monthly
)
from profile_agent import build_constraints

In [5]:
def candidate_rent(persona):
    c = build_constraints(persona['household'])['constraints']
    brma, _ = brma_for_district(c['area'])
    cat = LHA_CATEGORY.get(min(c['min_bedrooms'], 4), 'CAT E')
    return round(weekly_to_monthly(MARKET_30TH.loc[brma, cat]))


for p in personas[:5]:
    print(f"{p['id']} {p['label'][:30]:32s} £{candidate_rent(p)}")

P01a Single adult                     £1695
P01b Couple                           £1695
P02a Lone parent, one child           £1760
P02b Two sharers                      £1760
P03a Couple with two young sons       £1413


In [6]:
RIGHTS_QUESTIONS = [
    "Can my landlord refuse to rent to me because I have children?",
    "How much notice must my landlord give to end my tenancy?",
    "Can my landlord increase the rent during my tenancy?",
    "Am I allowed to keep a pet in a rented property?",
    "Can my landlord ask me to pay several months' rent in advance?",
]

def question_for(i):
    """Assign one tenancy question per persona, cycling through the set."""
    return RIGHTS_QUESTIONS[i % len(RIGHTS_QUESTIONS)]

In [7]:
def evaluate_persona(persona, rent, run_id, rights_question=None):
    """
    Run both architectures on one persona and record the outcome.

    Constraint satisfaction is assessed against ground truth computed
    independently: a recommendation satisfies the budget if total occupancy
    cost does not exceed the ceiling, and satisfies the bedroom floor if it
    does not propose fewer bedrooms than the statutory entitlement.
    """
    c = build_constraints(persona['household'])['constraints']
    rows = []

    # Ground truth, computed independently of either architecture's claim
    ct, _ = council_tax_for_district(c['area'])
    true_affordable = (rent + (ct or 0)) <= c['budget_max']

    # --- Multi-agent ---
    t0 = time.perf_counter()
    try:
        ma = recommend(persona['household'], rent, rights_question)
        ma_err = None
    except Exception as e:
        ma, ma_err = None, str(e)
    ma_time = time.perf_counter() - t0

    rows.append({
        'run_id': run_id,
        'persona': persona['id'],
        'pair_id': persona['pair_id'],
        'architecture': 'multi_agent',
        'completed': ma is not None,
        'error': ma_err,
        'latency_s': round(ma_time, 2),
        'model_calls': 0,
        'agents_called': ','.join(ma['agents_called']) if ma else None,
        'within_budget': ma['affordability']['within_budget'] if ma else None,
        'affordability_correct': (ma['affordability']['within_budget'] == true_affordable) if ma else None,
        'recommended_bedrooms': c['preferred_bedrooms'],
        'bedroom_violation': False,
        'min_bedrooms': c['min_bedrooms'],
        'expected_rent': ma['expected_rent'] if ma else None,
        'n_alternatives': (len(ma['alternatives']['cheaper_areas']) if ma and ma['alternatives'] else 0),
        'prompt_tokens': ma['tokens']['prompt_tokens'] if ma else 0,
        'output_tokens': ma['tokens']['output_tokens'] if ma else 0,
    })

    # --- Single agent ---
    t0 = time.perf_counter()
    try:
        sa = run_baseline(persona['household'], rent, rights_question)
        sa_err = None
    except Exception as e:
        sa, sa_err = None, str(e)
    sa_time = time.perf_counter() - t0

    sa_said = sa.get('affordable') if sa else None
    sa_beds = sa.get('recommended_bedrooms') if sa else None

    rows.append({
        'run_id': run_id,
        'persona': persona['id'],
        'pair_id': persona['pair_id'],
        'architecture': 'single_agent',
        'completed': bool(sa and sa['completed']),
        'error': sa_err or ('; '.join(sa['failures']) if sa else None),
        'latency_s': round(sa_time, 2),
        'model_calls': sa['steps'] if sa else 0,
        'agents_called': ','.join(sa['tools_called']) if sa else None,
        'within_budget': sa_said,
        'affordability_correct': (sa_said == true_affordable
                                  if sa_said is not None else None),
        'recommended_bedrooms': sa_beds,
        'bedroom_violation': (sa_beds < c['min_bedrooms']
                              if sa_beds is not None else None),
        'min_bedrooms': c['min_bedrooms'],
        'expected_rent': None,
        'n_alternatives': None,
        'prompt_tokens': sa['tokens']['prompt'] if sa else 0,
        'output_tokens': sa['tokens']['output'] if sa else 0,
    })

    return rows

In [8]:
#test run 1
'''p = personas[0]
rows = evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0))

for r in rows:
    print(r['architecture'], '|', r['completed'],
          '|', r['latency_s'], 's',
          '|', r['prompt_tokens'], 'prompt tokens',
          '|', r['output_tokens'], 'output tokens')'''

"p = personas[0]\nrows = evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0))\n\nfor r in rows:\n    print(r['architecture'], '|', r['completed'],\n          '|', r['latency_s'], 's',\n          '|', r['prompt_tokens'], 'prompt tokens',\n          '|', r['output_tokens'], 'output tokens')"

In [9]:
#test run 2
'''p = personas[0]
rows = evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0))

for r in rows:
    print(r['architecture'])
    print("  completed:  ", r['completed'])
    print("  latency:    ", r['latency_s'], "s")
    print("  affordability correct:", r['affordability_correct'])
    print("  bedrooms:   ", r['recommended_bedrooms'], "(floor", r['min_bedrooms'], ")")
    print("  violation:  ", r['bedroom_violation'])
    print()'''

'p = personas[0]\nrows = evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0))\n\nfor r in rows:\n    print(r[\'architecture\'])\n    print("  completed:  ", r[\'completed\'])\n    print("  latency:    ", r[\'latency_s\'], "s")\n    print("  affordability correct:", r[\'affordability_correct\'])\n    print("  bedrooms:   ", r[\'recommended_bedrooms\'], "(floor", r[\'min_bedrooms\'], ")")\n    print("  violation:  ", r[\'bedroom_violation\'])\n    print()'

In [10]:
#test run 3
'''subset = personas[:5]
results = []
for p in subset:
    results.extend(evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0)))

df = pd.DataFrame(results)
print(df[['persona', 'architecture', 'completed', 'latency_s', 'model_calls']].to_string(index=False))'''

"subset = personas[:5]\nresults = []\nfor p in subset:\n    results.extend(evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0)))\n\ndf = pd.DataFrame(results)\nprint(df[['persona', 'architecture', 'completed', 'latency_s', 'model_calls']].to_string(index=False))"

In [11]:
import time

REPETITIONS = 1
RUN_ID = 2          # run 0 and 1 is already saved

existing = pd.read_csv('../results/rq2_evaluation.csv')
all_rows = existing.to_dict('records')
start = time.time()

for i, p in enumerate(personas):
    all_rows.extend(evaluate_persona(p, candidate_rent(p), RUN_ID, question_for(i)))
    elapsed = (time.time() - start) / 60
    print(f"--- {i+1}/{len(personas)} complete, {elapsed:.0f} min elapsed ---")
    pd.DataFrame(all_rows).to_csv('../results/rq2_evaluation.csv', index=False)

print("Done. Total rows:", len(all_rows))

[0] search_act: The tenant is looking for a 1-bedroom flat in E3, and the candidate property is £1695/month, which i
[1] get_benchmarks: The tenant is looking for a 1-bedroom flat in E3, and the candidate property is £1695/month, which i
[2] predict_rent: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[3] find_nearby_districts: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[4] search_act: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[5] search_act: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[6] search_act: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[7] get_benchmarks: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[8] predict_rent: The tenant's budget is £1

In [12]:
df = pd.read_csv('../results/rq2_evaluation.csv')
print("Rows:", len(df), "| runs:", sorted(df['run_id'].unique()))
print()

summary = df.groupby(['architecture', 'run_id']).agg(
    completed=('completed', 'mean'),
    affordability_correct=('affordability_correct', 'mean'),
    bedroom_violations=('bedroom_violation', 'sum'),
    latency=('latency_s', 'mean'),
    prompt_tokens=('prompt_tokens', 'mean'),
).round(3)
print(summary.to_string())
print()

Rows: 240 | runs: [np.int64(0), np.int64(1), np.int64(2)]

                     completed affordability_correct bedroom_violations  latency  prompt_tokens
architecture run_id                                                                            
multi_agent  0           0.950                   1.0                  0   61.562       2309.425
             1           1.000                   1.0                  0   70.577       2435.800
             2           1.000                   1.0                  0   67.151       2435.800
single_agent 0           0.625                   0.4                  4  476.384      12297.475
             1           0.625                  0.32                  3  468.554      12226.900
             2           0.625                  0.32                  3  462.701      12226.900



In [13]:
overall = df.groupby('architecture').agg(
    completed=('completed', 'mean'),
    completed_sd=('completed', 'std'),
    affordability_correct=('affordability_correct', 'mean'),
    bedroom_violations=('bedroom_violation', 'sum'),
    latency=('latency_s', 'mean'),
    latency_sd=('latency_s', 'std'),
    prompt_tokens=('prompt_tokens', 'mean'),
).round(3)
print(overall.T.to_string())

architecture          multi_agent single_agent
completed                   0.983        0.625
completed_sd                0.129        0.486
affordability_correct         1.0     0.346667
bedroom_violations              0           10
latency                     66.43      469.213
latency_sd                 51.197       93.824
prompt_tokens            2393.675    12250.425


In [13]:
sa = df[df['architecture'] == 'single_agent']
print("Completed:", sa['completed'].sum(), "of", len(sa))
print("Affordability correct, of those completed:",
      sa[sa['completed']]['affordability_correct'].mean().round(3))
print("Bedroom violations:", sa['bedroom_violation'].sum())

Completed: 25 of 40
Affordability correct, of those completed: 0.4
Bedroom violations: 4


In [14]:
print(df[(df['architecture']=='multi_agent') & (~df['completed'])][['persona','error']].to_string())

   persona     error
34    P09b  'saving'
46    P12b  'saving'


In [14]:
print(df.groupby('architecture')['within_budget'].value_counts())

architecture  within_budget
multi_agent   True             48
              False            30
single_agent  False            42
              True              8
Name: count, dtype: int64


In [16]:
v = df[df['bedroom_violation'] == True]
print(v[['persona','min_bedrooms','recommended_bedrooms']].to_string())

   persona  min_bedrooms  recommended_bedrooms
17    P05a             2                   1.0
35    P09b             2                   1.0
47    P12b             3                   2.0
49     U01             1                   0.0


In [17]:
sa = df[df['architecture'] == 'single_agent']
pairs = sa[sa['pair_id'].notna()]
piv = pairs.pivot_table(index='pair_id', columns='persona',
                        values=['recommended_bedrooms', 'within_budget'],
                        aggfunc='first')
print(piv.to_string())

        recommended_bedrooms                                                                       within_budget                                                                                                
persona                 P01a P01b P02a P02b P03a P03b P04b P05a P07a P07b P08a P08b P09b P12a P12b          P01a   P01b   P02a  P02b   P03a   P03b   P04b   P05a  P07a   P07b   P08a   P08b   P09b   P12a   P12b
pair_id                                                                                                                                                                                                         
pair01                   1.0  3.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN          True  False    NaN   NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN    NaN    NaN    NaN    NaN
pair02                   NaN  NaN  2.0  2.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN           NaN    NaN  False  True    NaN    NaN    NaN    NaN   N

In [13]:
sa = df[(df['architecture'] == 'single_agent') & (df['pair_id'].notna())]
piv = sa.pivot_table(index=['pair_id', 'run_id'], columns='persona',
                     values='within_budget', aggfunc='first')
print(piv.to_string())

persona         P01a   P01b   P02a  P02b   P03a   P03b   P04b   P05a  P07a   P07b   P08a   P08b   P09b   P12a   P12b
pair_id run_id                                                                                                      
pair01  0       True  False    NaN   NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN    NaN    NaN    NaN    NaN
        1       True  False    NaN   NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN    NaN    NaN    NaN    NaN
pair02  0        NaN    NaN  False  True    NaN    NaN    NaN    NaN   NaN    NaN    NaN    NaN    NaN    NaN    NaN
        1        NaN    NaN  False  True    NaN    NaN    NaN    NaN   NaN    NaN    NaN    NaN    NaN    NaN    NaN
pair03  0        NaN    NaN    NaN   NaN  False  False    NaN    NaN   NaN    NaN    NaN    NaN    NaN    NaN    NaN
        1        NaN    NaN    NaN   NaN  False  False    NaN    NaN   NaN    NaN    NaN    NaN    NaN    NaN    NaN
pair04  0        NaN    NaN    NaN   NaN    NaN    NaN  False   

In [14]:
sa = df[df['architecture'] == 'single_agent']
flips = sa.groupby('persona')['completed'].nunique()
print("Personas whose completion changed across runs:", (flips > 1).sum(), "of 40")

Personas whose completion changed across runs: 0 of 40


In [15]:
diff = sa.groupby('persona').agg(n_affordable=('within_budget', 'nunique'),
                                 n_beds=('recommended_bedrooms', 'nunique'))
print(diff[(diff > 1).any(axis=1)].to_string())

         n_affordable  n_beds
persona                      
P09b                2       1
P12b                2       2


In [16]:
for p in ['P09b', 'P12b']:
    rows = sa[sa['persona'] == p][['run_id', 'within_budget', 'recommended_bedrooms', 'completed']]
    print(p)
    print(rows.to_string(index=False))
    print()

P09b
 run_id within_budget  recommended_bedrooms  completed
      0         False                   1.0       True
      1          True                   1.0       True
      2          True                   1.0       True

P12b
 run_id within_budget  recommended_bedrooms  completed
      0         False                   2.0       True
      1          True                   3.0       True
      2          True                   3.0       True



In [17]:
ma = df[df['architecture'] == 'multi_agent']
print(ma[ma['persona'].isin(['P09b','P12b'])][['persona','run_id','completed','error']].to_string(index=False))

persona  run_id  completed    error
   P09b       0      False 'saving'
   P12b       0      False 'saving'
   P09b       1       True      NaN
   P12b       1       True      NaN
   P09b       2       True      NaN
   P12b       2       True      NaN


## RQ3: matched-pair divergence

Compares the single agent's recommendations between matched personas. Within a
pair every field is held constant except household composition, so divergence in
pairs with identical statutory entitlement has no suitability justification
available.

Runs 1 and 2 only. Run 0 preceded a correction to the Alternatives agent and is
reported separately.

In [18]:
sa = df[(df['architecture'] == 'single_agent') &
        (df['run_id'].isin([1, 2])) &
        (df['pair_id'].notna())]

rows = []
for pid, grp in sa.groupby('pair_id'):
    members = grp.groupby('persona').agg(
        completed=('completed', 'first'),
        within_budget=('within_budget', 'first'),
        rec_beds=('recommended_bedrooms', 'first'),
        min_beds=('min_bedrooms', 'first'),
    )
    if len(members) < 2:
        rows.append({'pair': pid, 'status': 'incomplete',
                     'present': len(members)})
        continue

    a, b = members.index[0], members.index[1]
    rows.append({
        'pair': pid,
        'status': 'complete',
        'persona_a': a, 'persona_b': b,
        'entitlement_matched': members.loc[a, 'min_beds'] == members.loc[b, 'min_beds'],
        'budget_diverged': members.loc[a, 'within_budget'] != members.loc[b, 'within_budget'],
        'beds_diverged': members.loc[a, 'rec_beds'] != members.loc[b, 'rec_beds'],
    })

pairs_df = pd.DataFrame(rows)
print(pairs_df.to_string(index=False))

Task was destroyed but it is pending!
task: <Task pending name='Task-173' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\azmat\Projects\rentagent\venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-174' coro=<Kernel.shell_main() running at C:\Users\azmat\Projects\rentagent\venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\azmat\Projects\rentagent\venv\Lib\site-packages\zmq\eventloop\zmqstream.py:564]>
C:\Users\azmat\Projects\rentagent\venv\Lib\site-packages\pandas\core\groupby\groupby.py:1764: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  @final
Task was destroyed but it is pending!
task: <Task pending name='Task-174' coro=<Kernel.shell_main() running at C:\Users\azmat\Projects\rentagent\venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


  pair   status persona_a persona_b  entitlement_matched  budget_diverged  beds_diverged
pair01 complete      P01a      P01b                 True             True           True
pair02 complete      P02a      P02b                 True             True          False
pair03 complete      P03a      P03b                 True            False          False
pair04 complete      P04a      P04b                 True             True           True
pair05 complete      P05a      P05b                 True             True           True
pair06 complete      P06a      P06b                 True            False           True
pair07 complete      P07a      P07b                False             True          False
pair08 complete      P08a      P08b                False            False          False
pair09 complete      P09a      P09b                False             True           True
pair10 complete      P10a      P10b                False            False           True
pair11 complete      

In [19]:
ma = df[(df['architecture'] == 'multi_agent') &
        (df['run_id'].isin([1, 2])) &
        (df['pair_id'].notna())]

rows = []
for pid, grp in ma.groupby('pair_id'):
    members = grp.groupby('persona').agg(
        within_budget=('within_budget', 'first'),
        rec_beds=('recommended_bedrooms', 'first'),
        min_beds=('min_bedrooms', 'first'),
        expected_rent=('expected_rent', 'first'),
        n_alternatives=('n_alternatives', 'first'),
    )
    a, b = members.index[0], members.index[1]
    rows.append({
        'pair': pid,
        'entitlement_matched': members.loc[a, 'min_beds'] == members.loc[b, 'min_beds'],
        'budget_diverged': members.loc[a, 'within_budget'] != members.loc[b, 'within_budget'],
        'beds_diverged': members.loc[a, 'rec_beds'] != members.loc[b, 'rec_beds'],
        'alts_diverged': members.loc[a, 'n_alternatives'] != members.loc[b, 'n_alternatives'],
    })

print(pd.DataFrame(rows).to_string(index=False))

  pair  entitlement_matched  budget_diverged  beds_diverged  alts_diverged
pair01                 True            False          False          False
pair02                 True            False          False          False
pair03                 True            False          False          False
pair04                 True            False          False          False
pair05                 True            False          False          False
pair06                 True            False          False          False
pair07                False             True          False           True
pair08                False             True          False           True
pair09                False             True           True          False
pair10                False            False           True          False
pair11                False            False           True          False
pair12                False             True           True          False
